# ELLA VIDEO ENGINE — Wan 2.2 TI2V 5B

**Kaggle GPU notebook | Text-to-Video + Image-to-Video**

This notebook builds the runtime, downloads the Q4 model components, and generates a first MP4. It does not require a paid API key. GPU access is subject to Kaggle quotas.

In [ ]:
import os, subprocess, shutil, json, textwrap, pathlib, time
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert shutil.which('nvidia-smi'), 'Enable a Kaggle GPU accelerator in Notebook settings.'
!apt-get update -qq && apt-get install -y -qq git cmake build-essential ffmpeg libopenblas-dev
!python -m pip -q install -U huggingface_hub


## 1. Build stable-diffusion.cpp with CUDA
Build once per fresh Kaggle session. The source code is fetched from its upstream repository; model weights are downloaded separately below.

In [ ]:
%cd /kaggle/working
if not pathlib.Path('stable-diffusion.cpp/.git').exists():
    !git clone --depth 1 --recurse-submodules https://github.com/leejet/stable-diffusion.cpp.git
%cd /kaggle/working/stable-diffusion.cpp
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DSD_CUDA=ON -DSD_CUDA_CUTLASS_PATH:PATH=
!cmake --build build --config Release -j2
SDCLI = '/kaggle/working/stable-diffusion.cpp/build/bin/sd-cli'
assert pathlib.Path(SDCLI).exists(), 'Build did not produce sd-cli; inspect the cell output.'
print('ELLA runtime built:', SDCLI)


## 2. Download the model components (Q4)
The generator alone is not enough: Wan also requires its matching VAE and UMT5 text encoder.

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_DIR = pathlib.Path('/kaggle/working/ella_models'); MODEL_DIR.mkdir(exist_ok=True)
def get_file(repo, name):
    target = MODEL_DIR / name
    if not target.exists():
        hf_hub_download(repo_id=repo, filename=name, local_dir=str(MODEL_DIR))
    assert target.exists(), f'Missing download: {target}'
    print(name, round(target.stat().st_size/1e9, 2), 'GB')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'Wan2.2-TI2V-5B-Q4_K_S.gguf')
get_file('off-grid-ai/Wan2.2-TI2V-5B-GGUF', 'wan2.2_vae.safetensors')
get_file('off-grid-ai/Video-Companions-GGUF', 'umt5-xxl-encoder-Q4_K_M.gguf')


## 3. Your video settings
Start with five seconds, 8 FPS and 1280×704. If the T4 runs out of memory, keep the error log and we will tune the exact setting rather than silently restarting.

In [ ]:
PROMPT = 'A cinematic realistic scene: a young woman walks slowly through a rainy Istanbul street at night, reflections on the pavement, natural human movement, dramatic film lighting, smooth camera tracking.'
NEGATIVE_PROMPT = 'low quality, blurry, distorted anatomy, extra fingers, flicker, subtitles, watermark, static frame'
MODE = 'text'  # use 'image' for image-to-video
REFERENCE_IMAGE = '/kaggle/working/reference.png'  # upload a file with this exact name for MODE='image'
DURATION_SECONDS = 5
FPS = 8
WIDTH, HEIGHT = 1280, 704
STEPS = 20
SEED = 42
print('Settings ready. Mode:', MODE)


In [ ]:
!mkdir -p /kaggle/working/ella_engine
print('ELLA runner will be fetched from the project branch in the next cell.')


## 4. Generate MP4
This cell calls the ELLA runner with argument-safe subprocess execution (no shell interpolation).

In [ ]:
import importlib.util
RUNNER_PATH = pathlib.Path('/kaggle/working/ella_engine/runner.py')
if not RUNNER_PATH.exists():
    import urllib.request
    url='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/ella-wan22-engine/services/ella-wan-engine/runner.py'
    urllib.request.urlretrieve(url, RUNNER_PATH)
spec=importlib.util.spec_from_file_location('ella_runner', RUNNER_PATH)
runner=importlib.util.module_from_spec(spec); spec.loader.exec_module(runner)
args=runner.parser().parse_args(['--prompt', PROMPT, '--negative-prompt', NEGATIVE_PROMPT, '--duration', str(DURATION_SECONDS), '--fps', str(FPS), '--width', str(WIDTH), '--height', str(HEIGHT), '--steps', str(STEPS), '--seed', str(SEED), '--output', '/kaggle/working/ella_output.mp4'])
args.sd_cli=pathlib.Path(SDCLI)
args.diffusion_model=MODEL_DIR/'Wan2.2-TI2V-5B-Q4_K_S.gguf'
args.vae=MODEL_DIR/'wan2.2_vae.safetensors'
args.text_encoder=MODEL_DIR/'umt5-xxl-encoder-Q4_K_M.gguf'
if MODE == 'image':
    args.image=pathlib.Path(REFERENCE_IMAGE)
result=runner.run(args)
print(json.dumps(result, ensure_ascii=False, indent=2))
from IPython.display import Video, display
display(Video('/kaggle/working/ella_output.mp4', embed=True))


## 5. Download your MP4
The final file is `/kaggle/working/ella_output.mp4`. Use the Kaggle file browser to download it. Keep the generation log (`ella_output.log`) if anything fails.

In [ ]:
!ls -lh /kaggle/working/ella_output.mp4 /kaggle/working/ella_output.log
